# Post-filter Attrition Analysis

Quantifies where candidates are failing in `lc_events_filtered` outputs and how threshold choices impact retained counts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('default')

In [ ]:
RUN_DIR = REPO_ROOT / 'output/runs/REPLACE_WITH_RUN'
CANDIDATES_FILE = RUN_DIR / 'results/lc_events_filtered.parquet'

if not CANDIDATES_FILE.exists():
    raise FileNotFoundError(f'Update CANDIDATES_FILE first: {CANDIDATES_FILE}')

if CANDIDATES_FILE.suffix.lower() == '.parquet':
    df = pd.read_parquet(CANDIDATES_FILE)
else:
    df = pd.read_parquet(CANDIDATES_FILE)

print('rows:', len(df))
failed_cols = [c for c in df.columns if c.startswith('failed_') and c != 'failed_any']
failed_cols

In [ ]:
attrition = pd.DataFrame({
    'filter': failed_cols,
    'n_failed': [int(pd.to_numeric(df[c], errors='coerce').fillna(0).astype(bool).sum()) for c in failed_cols],
})
attrition['frac_failed'] = attrition['n_failed'] / max(len(df), 1)
attrition = attrition.sort_values('n_failed', ascending=False).reset_index(drop=True)
display(attrition)

fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(attrition))))
ax.barh(attrition['filter'], attrition['n_failed'])
ax.invert_yaxis()
ax.set_xlabel('Failed candidates')
ax.set_title('Per-filter failure counts')
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative order analysis (edit order to test alternatives)
order = [
    'failed_posterior_strength',
    'failed_significant_detection',
    'failed_run_robustness',
    'failed_morphology',
    'failed_score',
    'failed_periodicity',
    'failed_gaia_ruwe',
    'failed_gaia_pm',
    'failed_periodic_catalog',
]
order = [c for c in order if c in df.columns]

remaining = pd.Series(True, index=df.index)
rows = []
for col in order:
    fail = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(bool)
    dropped = int((remaining & fail).sum())
    remaining &= ~fail
    rows.append({'step': col, 'dropped_here': dropped, 'remaining': int(remaining.sum())})
cum = pd.DataFrame(rows)
display(cum)

# Periodicity workload (checked subset proxy): rows with finite LSP products
if {'lsp_period', 'lsp_bootstrap_sig'}.issubset(df.columns):
    checked = np.isfinite(pd.to_numeric(df['lsp_period'], errors='coerce')) | np.isfinite(pd.to_numeric(df['lsp_bootstrap_sig'], errors='coerce'))
    print('periodicity checked:', int(checked.sum()), '/', len(df), f'({checked.mean():.1%})')

# Quick threshold sweeps for score/run_count style tuning
if {'dipper_score', 'jumper_score'}.issubset(df.columns):
    sweep = []
    for t in np.arange(-1.0, 2.01, 0.25):
        keep = (pd.to_numeric(df['dipper_score'], errors='coerce').fillna(-np.inf) >= t) | (pd.to_numeric(df['jumper_score'], errors='coerce').fillna(-np.inf) >= t)
        sweep.append({'score_threshold': float(t), 'kept': int(keep.sum()), 'frac_kept': float(keep.mean())})
    display(pd.DataFrame(sweep))